# 01 · Understand the prediction problem

**NFL Player Trajectory Lab** · Real 2023 tracking data · Training-only exploration

Predict each selected player's x/y position for every requested frame after the throw.
The competition supplies the ball landing point, target receiver, and forecast horizon.
This notebook reads the reproducible artifacts produced by `nfl benchmark`; it does
not silently retrain a model. Notebook 00 is optional orientation.

[Source and run instructions](https://github.com/alvaromendizabal/nfl-player-trajectory)

**Result of the completed experiment:** landing-aware residual ridge achieved 0.9269 coordinate RMSE. The analysis below distinguishes training associations from validation evidence.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
LOCAL = ROOT / "artifacts" / "benchmark"
PUBLISHED = ROOT / "docs" / "results"
RESULTS = LOCAL if (LOCAL / "summary.json").exists() else PUBLISHED
READY = (RESULTS / "summary.json").exists()
if READY:
    summary = json.loads((RESULTS / "summary.json").read_text())
    eda = json.loads((RESULTS / "eda.json").read_text())
    protocol = json.loads((RESULTS / "protocol.json").read_text())
    display(Markdown("**Report source:** " + ("local benchmark artifacts" if RESULTS == LOCAL else "published reproducible experiment snapshot")))
else:
    display(Markdown(
        "Run `nfl benchmark` after the data audit to create the real-data results. "
        "No synthetic result is substituted here."
    ))

## Freeze time before fitting

Keep all frames, players and plays of a game in one partition. These dates are checked against game identifiers and locked before training. Holdout labels are excluded from the development pipeline.

In [ ]:
if READY:
    display(pd.DataFrame.from_dict(protocol["partitions"], orient="index").loc[["train", "validation", "holdout"]])
    display(Markdown("**Holdout evaluation:** " + summary["holdout_evaluation"]))

## Data coverage

Input rows describe observed motion; target rows are the positions we score. A trajectory is one player within one play, and can contain several target frames.

In [ ]:
if READY:
    display(pd.DataFrame({"Training measure": ["Games", "Plays", "Scored trajectories", "Observed rows", "Target rows"], "Count": [eda[key] for key in ["games", "plays", "trajectories", "input_rows", "target_rows"]]}))
    display(pd.DataFrame(eda["weeks"])[["file", "games", "plays", "trajectories", "input_rows", "target_rows"]])

In [ ]:
if READY:
    display(Image(filename=str(RESULTS / "eda.png")))

## Candidate features, not unverified importance claims

The implemented bank has **2,843 deterministic candidate features**. It uses observed
history, landing-relative motion, nearest teammates/opponents, receiver and passer
anchors, and forecast-time interactions. Missing-history and missing-neighbour masks
are explicit. Names, birth dates, player IDs as numerical predictors, future coordinates,
and post-play outcomes are excluded.

`nfl features` screens candidates on **training residuals only**, drops constant and
near-duplicate training features, and retains at most **64 features per challenger**.
The motion-only, landing-aware, and interaction-aware ablations share a fixed ridge
regularization and feature budget. Validation chooses a development candidate; the
48-game holdout remains reserved. A large candidate count is not evidence of accuracy.

In [ ]:
from nfl_trajectory.features import feature_catalog

catalog = feature_catalog()
display(Markdown(f"**Candidate feature count:** {len(catalog):,}"))
display(catalog.groupby("family").size().rename("Candidate features").to_frame())

## Football hypotheses and the prediction boundary

The landing point and forecast horizon are supplied inputs, not post-throw player coordinates. Define the ball-relative vector `b = landing - last_position` and unit vector `u = b / max(||b||, epsilon)`. Radial speed is `v · u`; lateral speed is the signed 2-D cross product. These projections separate moving toward the ball from moving across its path.

| Signal | Football hypothesis | What must be checked |
|---|---|---|
| Radial and lateral motion × forecast time | Players decelerate and turn rather than extrapolate indefinitely. | Improvement at longer forecast horizons, not only short frames. |
| Receiver/defender relative position and velocity | Coverage depends on another player's route and separation. | Add-only comparisons that retain useful landing features. |
| Circular direction and orientation | Angles near 0° and 360° describe similar headings. | Rotation tests and missing-telemetry masks. |
| Multiscale pre-throw history | Recent cuts and longer route development carry different signals. | Training-only redundancy control and temporal validation. |
| Role and field geometry | Receivers and coverage defenders have different objectives. | Role-specific residuals; no player identity leakage. |

Feature construction uses actual frame gaps, not assumed contiguous rows. Context includes non-predicted players, with explicit availability indicators. Do not infer feature quality from the number of generated columns.

In [ ]:
import io

import matplotlib.pyplot as plt

from nfl_trajectory.research import load_evidence
from nfl_trajectory.runtime import Run

feature_summary, selection, evidence_label = load_evidence(ROOT)
display(Markdown(f"**Feature evidence:** {evidence_label}"))


def display_figure(figure):
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    plt.close(figure)
    display(Image(data=buffer.getvalue()))

## What the training data actually suggests

These are absolute training-residual associations, computed before validation. They are not causal importance and do not establish out-of-time benefit. The target is the remaining x/y error of the original role-ridge baseline; the larger absolute coordinate correlation is reported.

In [ ]:
with Run(ROOT, "training_relationships") as run:
    associations = pd.DataFrame(feature_summary["top_training_associations"])
    display(associations[["feature", "signal", "training_residual_correlation"]].head(12).round(4))
    plotted = associations.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 5.8), layout="constrained")
    labels = {
        "time_squared": "Forecast time squared",
        "time__lateral_speed": "Lateral speed × forecast time",
        "fraction__receiver1__dx": "Receiver forward gap × horizon fraction",
        "time__opponent1__closest_time": "Nearest opponent approach time × forecast time",
        "receiver1__dx": "Receiver forward gap",
        "fraction_squared__receiver1__dx": "Receiver forward gap × horizon fraction squared",
        "time__receiver1__dx": "Receiver forward gap × forecast time",
        "time__receiver1__ball_distance_advantage": "Receiver landing-distance advantage × time",
        "time__receiver1__closest_time": "Receiver approach time × forecast time",
        "time__ax": "Forward acceleration × forecast time",
    }
    ax.barh(plotted["feature"].map(labels), plotted["training_residual_correlation"])
    ax.set_xlabel("Absolute training residual correlation; not validation importance")
    ax.set_title("Time and ball-relative motion motivate the challenger", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    display_figure(fig)
    run.event("relationships_reviewed", training_rows=selection["training_rows"],
              candidate_features=len(catalog))

## Inspect the feature budget rather than expanding it blindly

A feature bank can be broad while the selected model remains redundant. The table below examines the exact fitted feature lists. Multiple `ball_ux` history terms can describe useful time scales, but their shared signal also motivates family-aware selection. Feature-name overlap is not a measured correlation coefficient.

In [ ]:
budget = pd.DataFrame({
    "Measure": ["Selected landing features", "Terms derived from ball_ux",
                "Features retained by both challengers", "Landing features displaced by interactions"],
    "Count": [selection["landing_selected_count"], selection["landing_ball_ux_count"],
              selection["overlap_count"], len(selection["removed_from_landing"])],
})
display(budget)
display(Markdown(
    f"**Observed:** {selection['landing_ball_ux_count']} of "
    f"{selection['landing_selected_count']} selected features derive from `ball_ux`. "
    "Test family redundancy before removing these features."
))

## Reproduce the feature experiment from this notebook

The execution switch is off for employer review. Enable it in the locked NFL kernel to run or resume the real experiment; existing verified stages are reused. The command streams timestamps, stage timings, and a heartbeat. It does not regenerate the original baseline, launch an instance, or submit to Kaggle. S3 checkpointing uses the existing private bucket.

In [ ]:
import subprocess
import sys

RUN_FEATURE_EXPERIMENT = False
if RUN_FEATURE_EXPERIMENT:
    subprocess.run(
        [sys.executable, "-m", "nfl_trajectory.cli", "features", "--checkpoint-s3"],
        cwd=ROOT, check=True,
    )
else:
    display(Markdown("Training switch is off; this notebook reviews completed real-data evidence."))